In [11]:
# Install required libraries
!pip install transformers sentencepiece

In [12]:
# Import required classes
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [13]:
# Load FLAN-T5 tokenizer
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")

# Load FLAN-T5 model
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

In [14]:
# Function to check unsafe medical questions
def safety_filter(user_input):

    # Dangerous keywords list
    dangerous_keywords = ["suicide", "kill", "overdose", "die"]

    # Convert user input to lowercase
    text = user_input.lower()

    # Check if any dangerous word exists in user input
    for word in dangerous_keywords:
        if word in text:
            return False

    # If no dangerous word found, input is safe
    return True

In [15]:
# Function to generate chatbot response
def ask_health_chatbot(user_input):

    # Check unsafe questions
    if not safety_filter(user_input):
        return "Please contact a healthcare professional or emergency service for serious medical concerns."

    # Create prompt for the model
    prompt = f"""
You are a helpful medical assistant.
Give a short, safe, and simple health-related answer.
Do not give harmful medical advice.
Recommend consulting a doctor when needed.

Question: {user_input}
Answer:
"""

    # Convert prompt into model input tokens
    inputs = tokenizer(prompt, return_tensors="pt")

    # Generate answer from model
    outputs = model.generate(
        **inputs,
        max_new_tokens=100
    )

    # Convert output tokens into readable text
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    # Return final answer
    return answer

In [16]:
# Start chatbot loop
while True:

    # Take question from user
    question = input("You: ")

    # Stop chatbot if user types exit
    if question.lower() == "exit":
        print("Chatbot: Goodbye!")
        break

    # Generate chatbot answer
    answer = ask_health_chatbot(question)

    # Print answer
    print("Chatbot:", answer)

You: What causes sore throat?
Chatbot: a viral infection
You: Is paracetamol safe for children?
Chatbot: Yes.
You: How can I reduce stress?
Chatbot: Eat a healthy diet.
You: exit
Chatbot: Goodbye!
